Read data from silver layer for all the tables.

In [0]:
silver_layer_path = "/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Silver/"

In [0]:
silver_customers_df = spark.read.format("delta").load(
    f"{silver_layer_path}/customers"
)

silver_orders_df = spark.read.format("delta").load(
    f"{silver_layer_path}/orders"
)

silver_order_items_df = spark.read.format("delta").load(
    f"{silver_layer_path}/order_items"
)

silver_products_df = spark.read.format("delta").load(
    f"{silver_layer_path}/products"
)

In [0]:
silver_customers_df.count()

99441

In [0]:
silver_orders_df.count()

99441

In [0]:
silver_order_items_df.count()

112650

In [0]:
silver_products_df.count()

32951

now will start joining multiple tables


In [0]:
gold_orders_customers_df = silver_orders_df.join(
    silver_customers_df,
    on="customer_id",
    how="left"
)
gold_orders_customers_df.count()

99441

In [0]:
gold_orders_customers_items_df = gold_orders_customers_df.join(
    silver_order_items_df,
    on="order_id",
    how="left"
)

gold_orders_customers_items_df.count()

113425

In [0]:
from pyspark.sql.functions import col
gold_orders_customers_items_df.filter(
    col("product_id").isNull()
).count()

775

joining all 3(Combined) df (custmer_orders_items_df) with silver_product_df 

In [0]:
gold_final_df = gold_orders_customers_items_df.join(
    silver_products_df,
    on="product_id",
    how="left"
)
gold_final_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- order_purchase_date: date (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- seller_id: string (nullable = tr

1)Calculating total sales
- so for that first generating total_item_value (means total amount including shipping a customer paid to buy a particular item)


In [0]:
gold_final_df=gold_final_df.withColumn(
    "total_item_value",
    col("price") + col("freight_value")
)
gold_final_df.select(
    "order_id",
    "product_id",
    "price",
    "freight_value",
    "total_item_value"
).limit(20).display()

order_id,product_id,price,freight_value,total_item_value
e481f51cbdc54678b7cc49136f2d6af7,87285b34884572647811a353c7ac498a,29.99,8.72,38.71
53cdb2fc8bc7dce0b6741e2150273451,595fac2a385ac33a80bd5114aec74eb8,118.7,22.76,141.46
47770eb9100c2d0c44946d9cf07ec65d,aa4383b373c6aca5d8797843e5594415,159.9,19.22,179.12
949d5b44dbf5de918fe9c16f97b45f8a,d0b61bfb1de832b15ba9d266ca96e5b0,45.0,27.2,72.2
ad21c59c0840e6cb83a9ceb5573f8159,65266b2da20d04dbe00c5c2d3bb7859e,19.9,8.72,28.619999999999997
a4591c265e18cb1dcee52889e2d8acc3,060cb19345d90064d1015407193c233d,147.9,27.36,175.26
136cce7faa42fdb2cefd53fdc79a6098,a1804276d9941ac0733cfd409f5206eb,49.9,16.05,65.95
6514b8ad8028c9f2cc2374ded245783f,4520766ec412348b8d4caa5e8a18c464,59.99,15.17,75.16
76c6e866289321a7c93b82b54852dc33,ac1789e492dcd698c5c10b97a671243a,19.9,16.05,35.95
e69bfb5eb88e0ed6a785585b27e16dbf,9a78fb9862b10749a117f7fc3c31f051,149.99,19.77,169.76000000000002


Generating total revenue of all the items(all the orders)

In [0]:
from pyspark.sql.functions import sum,round

total_revenue_generated = gold_final_df.select(
    round(sum("total_item_value"),2).alias("total_revenue")
)

In [0]:
gold_final_df.limit(10).display()

product_id,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,source_file,ingestion_timestamp,order_purchase_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,source_file,ingestion_timestamp,order_item_id,seller_id,shipping_limit_date,price,freight_value,source_file,ingestion_timestamp,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,source_file,ingestion_timestamp,total_item_value
87285b34884572647811a353c7ac498a,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_orders_dataset.csv,2026-09-20T10:29:58.088Z,2017-10-02,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:53.494Z,1,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06T11:07:15.000Z,29.99,8.72,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_order_items_dataset.csv,2026-09-20T10:30:01.096Z,utilidades_domesticas,40,268,4,500,19,8,13,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z,38.71
595fac2a385ac33a80bd5114aec74eb8,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_orders_dataset.csv,2026-09-20T10:29:58.088Z,2018-07-24,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:53.494Z,1,289cdb325fb7e7f891c38608bf9e0962,2018-07-30T03:24:27.000Z,118.7,22.76,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_order_items_dataset.csv,2026-09-20T10:30:01.096Z,perfumaria,29,178,1,400,19,13,19,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z,141.46
aa4383b373c6aca5d8797843e5594415,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_orders_dataset.csv,2026-09-20T10:29:58.088Z,2018-08-08,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:53.494Z,1,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13T08:55:23.000Z,159.9,19.22,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_order_items_dataset.csv,2026-09-20T10:30:01.096Z,automotivo,46,232,1,420,24,19,21,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv,2026-09-20T10:30:03.476Z,179.12
d0b61bfb1de832b15ba9d266ca96e5b0,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_orders_dataset.csv,2026-09-20T10:29:58.088Z,2017-11-18,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/oli

q2) find out average amount a customer spent on one order,
- so in one order he can order multiple products so sum of price of each product of one order then / total number of orders is the avg of total orders spent by customer 
- ans ==> it means on an average a customer spent 160.58(brazilian dollar) per order

In [0]:
from pyspark.sql.functions import sum, avg, round
order_value_df = gold_final_df.groupBy("order_id").agg(
    sum("total_item_value").alias("order_value")
)
order_value_df.select(
    round(avg("order_value"),2).alias("average")
).display()

average
160.58


q3) Which product categories generate the most revenue?
- product id and category in product table 
and product_it, price and freight_value is in order_item table
we have alredy combined it in gold_final_df, 

- solution will group by product_category and add(total_value) for each product and will arrange in desc order to find most generated revenue by a product
- so after query answer is beleza_saude category of product is generate most revenue with approax 1.44million


In [0]:
from pyspark.sql.functions import desc
product_category_revenue_df = gold_final_df.groupBy("product_category_name").agg(
    round(sum("total_item_value"),2).alias('total_revenue')
).orderBy(
    desc('total_revenue')
)

q4) which particular product has been sold for most of the time.

In [0]:
from pyspark.sql.functions import desc, count, col

most_sold_product_df = gold_final_df.filter(
    col("product_id").isNotNull()
).groupBy("product_id").agg(
    count("*").alias("each_product_total_units_sold")
).orderBy(
    desc('each_product_total_units_sold')
)
most_sold_product_df.limit(5).display()

product_id,each_product_total_units_sold
aca2eb7d00ea1a7b8ebd4e68314663af,527
99a4788cb24856965c36a24e339b6058,488
422879e10f46682990de24d770e7f83d,484
389d119b48cf3043d311335e499d9c6b,392
368c6c730842d78016ad823897a372db,388


q5)Monthly Sales Trend
- How much revenue did we generate in each month?

In [0]:
from pyspark.sql.functions import year, month, sum, round

monthly_sales_df = gold_final_df.withColumn(
    "year",
    year("order_purchase_date")
).withColumn(
    "month",
    month("order_purchase_date")
).groupBy(
    "year", "month"
).agg(
    round(sum("total_item_value"),2).alias("monthly_revenue")
).orderBy(desc('year'), 'month')

q6) order status distribution
- how many orders in each status 
- like 
Status          Number of Orders
delivered       96,478
shipped          1,107
canceled           625

- look at the diff b/w count() and countDistinct()

In [0]:
from pyspark.sql.functions import desc, countDistinct

monthly_sales_df = gold_final_df.withColumn(
    "year",
    year("order_purchase_date")
).withColumn(
    "month",
    month("order_purchase_date")
)
gold_final_df.groupBy('order_status').agg(
    countDistinct("order_id").alias('order_count')
).orderBy(desc('order_count')).display()

order_status,order_count
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


q7) Which customer states(like mp, mh) generate the most revenue
- Which Brazilian state contributes the most revenue to the business?

In [0]:
state_revenue_df = gold_final_df.groupBy("customer_state").agg(
    round(sum('total_item_value'),2).alias('total_revenue_by_state')
).orderBy(desc('total_revenue_by_state'))


q8) how many orders does each customer made

In [0]:
from pyspark.sql.functions import countDistinct, desc, count

customer_orders_df = gold_final_df.groupBy(
    "customer_id"
).agg(
    countDistinct("order_id").alias("order_count")
).orderBy(
    desc("order_count")
)

customer_orders_df.limit(10).display()

customer_id,order_count
f54a9f0e6b351c431402b8461ea51999,1
503740e9ca751ccdda7ba28e9ab8f608,1
ed0271e0b7da060a393796590e7b737a,1
9bdf08b4b3b52b5526ff42d37d47f222,1
b0830fb4747a6c6d20dea0b8c802d7ef,1
8ab97904e6daea8866dbdbc4fb7aad2c,1
9ef432eb6251297304e76186b10a928d,1
f88197465ea7920adcdbec7375364d82,1
41ce2a54c0b03bf3443c3d931a367089,1
31ad1d1b63eb9962463f764d4e6e0c9d,1


9)Who are the top customers by spending?
- We want to find customers who spent the most money.


In [0]:
from pyspark.sql.functions import sum, round, desc

top_customers_df = gold_final_df.groupBy(
    "customer_unique_id"
).agg(
    round(sum("total_item_value"), 2).alias("total_spent")
).orderBy(
    desc("total_spent")
)

top_customers_df.limit(10).display()

customer_unique_id,total_spent
0a0a92112bd4c708ca5fde585afaa872,13664.08
da122df9eeddfedc1dc1f5349a1a690c,7571.63
763c8b1c9c68a0229c42c9fc6f662b93,7274.88
dc4802a71eae9be1dd28f5d788ceb526,6929.31
459bef486812aa25204be022145caa62,6922.21
ff4159b92c40ebe40454e3e6a7c35ed6,6726.66
4007669dec559734d6f53e029e360987,6081.54
5d0a2980b292d049061542014e8960bf,4809.44
eebb5dda148d3893cdaf5b5ca3040ccb,4764.34
48e1ac109decbb87765a3eade6854098,4681.78


10) What is the average delivery time?
- We calculate the time between order purchase and actual delivery.
- after querying ==> used 86,400 becoz a day has 24hrs, 
1hr = 60mins, and 1min = 60secs
so total ==> 24x60x60=86,400


In [0]:
from pyspark.sql.functions import col, avg, round

delivery_time_df = gold_final_df.filter(
    col("order_delivered_customer_date").isNotNull()
).withColumn(
    "delivery_days",
    (col("order_delivered_customer_date").cast("long") -
     col("order_purchase_timestamp").cast("long")) / 86400
)

delivery_time_df.select(
    round(avg("delivery_days"), 2).alias("average_delivery_days")
).display()

average_delivery_days
12.47


bussiness logical questions are completed, now moving towards storing data into gold layer.

In [0]:
golden_layer_path = silver_layer_path.replace("Silver", "Golden")

1)sales summary 
- this is what we wnat in sales summary
sales_summary
├── total_revenue
├── average_order_value
└── total_orders

In [0]:
from pyspark.sql.functions import sum

order_value_df = gold_final_df.groupBy(
    "order_id"
).agg(
    sum("total_item_value").alias("order_value")
)
order_value_df.limit(10).display()

order_id,order_value
432aaf21d85167c2c86ec9448c4e42cc,54.36
85ce859fd6dc634de8d2f1e290444043,29.75
734e7d1bbaeb2ff82521ca0fe6fb6f79,43.46
e346cd9299371b18c0b28e8e29a5e376,42.589999999999996
a5474c0071dd5d1074e12d417078bbd0,21.38
4b3a605942f29d490cb74bd6ace6b9f0,38.099999999999994
90349f264a3d6a2525a34598d09dda6b,314.34
93ec3e2c9a4beee38c28973d307093e1,204.45000000000002
21c71f62d2554e1ad6c8ba9dc7af2c62,74.16
7156f3e0b94405abcbad1a8c94557338,28.46


answering above 3 questions.


In [0]:
from pyspark.sql.functions import sum, avg, countDistinct, round

sales_summary_df = order_value_df.agg(
    round(sum("order_value"), 2).alias("total_revenue"),
    round(avg("order_value"), 2).alias("average_order_value"),
    countDistinct("order_id").alias("total_orders")
)
sales_summary_df.display()

total_revenue,average_order_value,total_orders
1.584355324E7,160.58,99441


In [0]:
sales_summary_df.write.format("delta").mode("overwrite").save(f"{golden_layer_path}/sales_summary")

In [0]:
sales_summary_gold_df = spark.read.format("delta").load(f"{golden_layer_path}/sales_summary")
sales_summary_gold_df.display()

total_revenue,average_order_value,total_orders
1.584355324E7,160.58,99441


saving product_category_revenue


In [0]:
product_category_revenue_df.write.format("delta").mode("overwrite").save(f"{golden_layer_path}/category_revenue_df")

In [0]:
spark.read.format("delta").load(f"{golden_layer_path}/category_revenue_df").limit(20).display()

product_category_name,total_revenue
beleza_saude,1441248.07
relogios_presentes,1305541.61
cama_mesa_banho,1241681.72
esporte_lazer,1156656.48
informatica_acessorios,1059272.4
moveis_decoracao,902511.79
utilidades_domesticas,778397.77
cool_stuff,719329.95
automotivo,685384.32
ferramentas_jardim,584219.21


In [0]:
monthly_sales_df.write.format("delta").mode("overwrite").save(f"{golden_layer_path}/monthly_sales_revenue")
spark.read.format("delta").load(f"{golden_layer_path}/monthly_sales_revenue").limit(20).display()

year,month,monthly_revenue
2018,1,1107301.89
2018,2,986908.96
2018,3,1155126.82
2018,4,1159698.04
2018,5,1149781.82
2018,6,1022677.11
2018,7,1058728.03
2018,8,1003308.47
2018,9,166.46
2018,10,null


In [0]:
state_revenue_df.write.format("delta").mode("overwrite").save(f"{golden_layer_path}/state_revenue")
spark.read.format("delta").load(f"{golden_layer_path}/state_revenue").display()

customer_state,total_revenue_by_state
SP,5921678.12
RJ,2129681.98
MG,1856161.49
RS,885826.76
PR,800935.44
BA,611506.67
SC,610213.6
DF,353229.44
GO,347706.93
ES,324801.91


In [0]:
(top_customers_df.write 
    .format("delta") 
    .mode("overwrite") 
    .save(f"{golden_layer_path}/top_customers_spending")
)
spark.read.format("delta").load(f"{golden_layer_path}/top_customers_spending").limit(20).display()

customer_unique_id,total_spent
0a0a92112bd4c708ca5fde585afaa872,13664.08
da122df9eeddfedc1dc1f5349a1a690c,7571.63
763c8b1c9c68a0229c42c9fc6f662b93,7274.88
dc4802a71eae9be1dd28f5d788ceb526,6929.31
459bef486812aa25204be022145caa62,6922.21
ff4159b92c40ebe40454e3e6a7c35ed6,6726.66
4007669dec559734d6f53e029e360987,6081.54
5d0a2980b292d049061542014e8960bf,4809.44
eebb5dda148d3893cdaf5b5ca3040ccb,4764.34
48e1ac109decbb87765a3eade6854098,4681.78
